# Calculate product unit footprints for all available Ecoinvent locations

## Notebook README

**Related publication**

If you utilize any portions of the code, results, or draw inspiration for your projects, please give proper credit by citing the published article below.

 - Title: Residual biomass to bio-based chemicals and plastics: ex-ante screening methodology for prioritizing high-impact substitutions
 - Authors: [Nicolas LIENART](https://orcid.org/0009-0001-3259-2819), [Thibaut LECOMPTE](https://orcid.org/0000-0001-9237-8454), [Lorie HAMELIN](https://orcid.org/0000-0001-9092-1900) 
 - Journal: Resources, Conservation and Recycling (RCR) - Elsevier
 - Doi: #todo
 - Code author: Nicolas LIENART
 - Git repository (Forge INRAE): https://forge.inrae.fr/nicolas.lienart/screen-lca-paper-supplementary-code
 - Git repository (GitHub): https://github.com/nicolnt/screen-lca-paper-supplementary-code

**Description and details**

This notebook is used as part of the sensitivity analysis. It calculates the unit product footprints for each selected products and the available locations in the different Ecoinvent versions. 

**Package versions**

 - See [`environment.yml`](./environment.yml)

**Licence**

This work is licenced under the Creative Commons Attribution (CC-BY 4.0) public licence.

## Initialization

### Python imports

In [1]:
import pandas as pd
from importlib.metadata import version

In [2]:
print('pandas:', version('pandas'))

pandas: 2.2.3


In [3]:
# NOTE: Local import
import sys
sys.path.append("../")

from Python_utils import brightway_database

20:31:01+0200 [warning  ] Can't import `SimaProBlockCSVImporter` - please install `bw2io` with `pip install bw2io[multifunctional]` or install `multifunctional` and `bw_simapro_csv` manually.


### Import some global variables from file

Your `.env` file is a collection of key-value pairs, separated by a `=` sign. It should contain these lines with the relevant values:

```bash
project_name="replace with project name"
```

We can then access this information with the dotenv and Python's built-in `os` libraries.

In [4]:
import os
from dotenv import load_dotenv  # To read the contents of the .env file

load_dotenv(override=True)

BW_PROJECT_NAME = os.getenv("project_name")

In [5]:
# NOTE: Manually set Brightway current project
# PROJECT_NAME = "a-brightway-project"

### Manually set other global variables

In [6]:
ECOINVENT_ACTIVITY_COLUMN_NAME = "Ecoinvent activity name or proxy"

In [7]:
ECOINVENT_DATABASE_NAMES = ['ecoinvent-3.11-cutoff', 'ecoinvent-3.12-cutoff']

In [8]:
ECOINVENT_DATABASES = {}

for ecoinvent_database_name in ECOINVENT_DATABASE_NAMES:
    # NOTE: Produces a string which looks something like "ecoinvent-x.x.x" out of "ecoinvent-x.x.x-model"
    method_key_db_name = "-".join(ecoinvent_database_name.split("-")[0:2])

    # NOTE: Select the relevant key, based on Ecoinvent version (no matter the system model)
    gwp_key = (method_key_db_name, "IPCC 2021", "climate change: total (excl. biogenic CO2)", "global warming potential (GWP100)")

    ECOINVENT_DATABASES[ecoinvent_database_name] = brightway_database.BrightwayDatabase(bw_database_name=ecoinvent_database_name, bw_project_name=BW_PROJECT_NAME, method_key=gwp_key)

# ECOINVENT_DATABASES = ecoinvent_databases.get_ecoinvent_databases(ECOINVENT_DATABASE_NAMES, project_name=BW_PROJECT_NAME)
ECOINVENT_DATABASES

{'ecoinvent-3.11-cutoff': <Python_utils.brightway_database.BrightwayDatabase at 0x7f2713705940>,
 'ecoinvent-3.12-cutoff': <Python_utils.brightway_database.BrightwayDatabase at 0x7f27136c7610>}

### Additional settings

In [9]:
pd.options.display.max_columns = 100
pd.options.display.max_rows = 5

## Import Prodcom hotspot products for footprint calculation

In [10]:
hotspot_products_df = pd.read_csv('../Output_data/Hotspot products export - with ecoinvent regions.csv', index_col=0, dtype={
    'PRODCOM code': str,
    'HS22 code': str,
})

In [11]:
# NOTE: Remove unused columns from the CSV, remove listed columns below
hotspot_products_with_footprint_df = hotspot_products_df.loc[:, hotspot_products_df.columns.isin([ECOINVENT_ACTIVITY_COLUMN_NAME])].copy()

## Calculate footprints

In [13]:
unit_footprint_df = pd.DataFrame(data=[], index=pd.Index(data=[], name="activity"), columns=pd.MultiIndex(levels=[[], []], codes=[[], []], names=["ecoinvent_database_name", "location"]))


for ecoinvent_database_name in ECOINVENT_DATABASE_NAMES:
    ecoinvent_db: brightway_database.BrightwayDatabase = ECOINVENT_DATABASES[ecoinvent_database_name]

    for hotspot_product_index, hotspot_product in hotspot_products_df.iterrows():
        ecoinvent_activity_name = hotspot_product[ECOINVENT_ACTIVITY_COLUMN_NAME]

        activities = [a for a in ecoinvent_db.db.search(ecoinvent_activity_name) if a['name'] == ecoinvent_activity_name]
        
        for activity in activities:
            location = activity['location']
            id = activity.id
            score = ecoinvent_db.calculate_LCA_optimized(activity_id=id)
            unit_footprint_df.loc[ecoinvent_activity_name, (ecoinvent_database_name, location)] = score

In [16]:
unit_footprint_df.index

Index(['market for polypropylene, granulate', 'market for ethylene',
       'market for propylene',
       'market for polyethylene, high density, granulate',
       'market for methanol', 'market for benzene',
       'market for polyvinyl chloride, emulsion polymerised',
       'market for polyethylene, low density, granulate',
       'market for polyethylene, linear low density, granulate',
       'market for styrene', 'market for purified terephthalic acid',
       'market for polyethylene terephthalate, granulate, amorphous',
       'market for butadiene', 'market for butene, mixed',
       'market for polystyrene, expandable', 'market for ethylene glycol',
       'market for propylene oxide, liquid',
       'market for polystyrene, extruded', 'market for ethylene dichloride',
       'market for p-xylene', 'market for toluene, liquid',
       'market for acetone, liquid', 'market for formaldehyde',
       'market for vinyl acetate', 'market for acetic acid',
       'market for ethy

In [17]:
unit_footprint_df.columns

MultiIndex([('ecoinvent-3.11-cutoff',                'GLO'),
            ('ecoinvent-3.11-cutoff',                 'CN'),
            ('ecoinvent-3.11-cutoff',                'RoW'),
            ('ecoinvent-3.11-cutoff',                 'ZA'),
            ('ecoinvent-3.11-cutoff',                'RER'),
            ('ecoinvent-3.11-cutoff',                 'US'),
            ('ecoinvent-3.12-cutoff',                'RoW'),
            ('ecoinvent-3.12-cutoff',                'RAF'),
            ('ecoinvent-3.12-cutoff',                'RNA'),
            ('ecoinvent-3.12-cutoff',                 'CN'),
            ('ecoinvent-3.12-cutoff',                'RLA'),
            ('ecoinvent-3.12-cutoff',                'RER'),
            ('ecoinvent-3.12-cutoff', 'Asia without China'),
            ('ecoinvent-3.12-cutoff',                 'QA'),
            ('ecoinvent-3.12-cutoff',                 'TW'),
            ('ecoinvent-3.12-cutoff',                 'AE'),
            ('ecoinvent-

In [18]:
unit_footprint_df.to_csv("../Output_data/Ecoinvent unit footprints.csv")